# 02 — Baseline evaluation of SmolLM2-360M

Measure the untouched model before fine-tuning. This notebook reuses the frozen data, tokenizer, model revision, and prompts from Notebook 01.

**Before Run All:** enable **Internet**, select a **GPU** accelerator, and use **Add Input** to attach the saved output of Notebook 01. Attach one data bundle. A ZIP uploaded as a Kaggle dataset also works.

The workflow runs 20 validation questions in both prompt modes (40 smoke predictions), estimates generation time, then evaluates all 2,819 test questions in both modes (5,638 predictions). Run All includes the full benchmark and consumes GPU time. To review the estimate first, execute cells only through the smoke-test section, then continue.

Predictions are saved after every batch. An interrupted run can resume with the same data, code, environment, hardware, and settings. No fine-tuning is performed here.

## 1. Fetch the GitHub code

Each run downloads the current `main` branch and records its commit. To reproduce an older run, `MATH_PROJECT_REF` can select its recorded commit. Shared code updates arrive through GitHub without copying cells.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)

## 2. Check GPU allocation and find the saved data bundle

Attach Notebook 01's saved output using Kaggle **Add Input**. This cell searches `/kaggle/input` and refuses to choose silently if several bundles are attached. Optional `MATH_BUNDLE_PATH` can identify a particular bundle directory or ZIP.

Only the saved bundle is used: this notebook does not regenerate splits or demonstrations. It verifies all recorded data hashes before model setup. Run directories are stable for the bundle, code commit, and batch size so rerunning can resume saved work.

In [ ]:
import shutil
import hashlib

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a GPU accelerator in Kaggle notebook settings, then start the session again.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
INPUT_ROOT = Path(os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input"))
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), str(INPUT_ROOT)], text=True).strip())
subprocess.run([sys.executable, str(PROJECT / "scripts/verify_bundle.py"), str(BUNDLE)], check=True)
DATA_MANIFEST = json.loads((BUNDLE / "manifest.json").read_text())
BUNDLE_HASH = hashlib.sha256((BUNDLE / "manifest.json").read_bytes()).hexdigest()
BATCH_SIZE = int(os.environ.get("MATH_BATCH_SIZE", "4"))
if BATCH_SIZE < 1:
    raise ValueError("Batch size must be positive")
RUN_ROOT = Path("/kaggle/working") / f"baseline-{BUNDLE_HASH[:12]}-{CODE_COMMIT[:8]}-batch{BATCH_SIZE}"
# Restore an attached previous notebook output only when its run directory matches.
if not RUN_ROOT.exists():
    previous = [p for p in Path("/kaggle/input").rglob(RUN_ROOT.name) if p.is_dir()]
    if len(previous) > 1:
        raise RuntimeError("Multiple matching previous runs attached; retain one to resume.")
    if previous:
        shutil.copytree(previous[0], RUN_ROOT)
        print("Restored prior predictions from:", previous[0])
print("Verified data bundle:", BUNDLE)
print("Frozen model:", DATA_MANIFEST["model"])
print("Output directory:", RUN_ROOT)
print("Batch size:", BATCH_SIZE)

## 3. Prepare and verify the inference environment

The shared setup script installs the same tokenizer stack and pinned **PyTorch 2.7.1 with CUDA 11.8** into an isolated environment. This CUDA build is chosen for compatibility with older GPU hardware as well as T4-class hardware; actual driver/kernel compatibility is checked below. Only the first GPU is used.

Setup may download several GB. It does not replace the notebook kernel's packages. The small matrix multiplication checks real GPU execution before any model weights are downloaded.

In [ ]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-baseline-env-"))
subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
                "--environment", str(ENVIRONMENT), "--profile", "baseline"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
probe = """
import torch
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot use CUDA. Check Kaggle GPU allocation and the driver/runtime setup.')
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA matrix multiplication passed')
"""
subprocess.run([str(PYTHON), "-I", "-c", probe], check=True)

## 4. Validation smoke test

Load the base model at Notebook 01's exact revision. Evaluate the first 20 validation questions in zero-shot and three-shot modes, using greedy decoding and the frozen 512-token output budget. A short unscored warmup runs first.

This checks inference, scoring, memory, and saving. A low accuracy is possible for this small base model and does not itself indicate a broken pipeline. Answers without exactly one parseable `Final answer:` line count as invalid and incorrect. We retain raw completions for analysis.

In [ ]:
SMOKE_OUTPUT = RUN_ROOT / "smoke"
subprocess.run([str(PYTHON), str(PROJECT / "scripts/run_baseline.py"),
                "--bundle", str(BUNDLE), "--output", str(SMOKE_OUTPUT), "--stage", "smoke",
                "--batch-size", str(BATCH_SIZE), "--smoke-count", "20"], check=True, cwd=PROJECT)
SMOKE = json.loads((SMOKE_OUTPUT / "summary.json").read_text())
from IPython.display import display, HTML, Markdown

display(HTML((SMOKE_OUTPUT / "report.html").read_text()))
TEST_QUESTIONS = sum(DATA_MANIFEST["counts"][name] for name in ["gsm8k_test", "svamp_test", "arithmetic_test"])
estimated_seconds = sum(g["mean_amortized_generation_seconds"] * TEST_QUESTIONS for g in SMOKE["groups"].values())
print(f"Approximate full-run generation time: {estimated_seconds / 60:.1f} minutes")
print("This estimate excludes installation/model loading and may change with question lengths and early stopping.")
print(f"Full benchmark: {TEST_QUESTIONS:,} questions × 2 prompt modes = {TEST_QUESTIONS * 2:,} predictions")

## 5. Full baseline evaluation

Run GSM8K test, SVAMP, and arithmetic questions in **both** prompt modes. Each original question receives the same settings planned for the later adapted-model evaluation. The three-shot demonstrations come from training only; reference test answers are used only after generation for scoring.

The script requires a successful smoke run on the same data/code/hardware. It saves each batch to `predictions.jsonl`, prints progress, and writes aggregate reports. Incomplete runs are marked partial. If interrupted, rerun this cell; saved predictions are reused. Do not change batch size, dependencies, GPU type, or code mid-run. If those must change, start a new run.

In [ ]:
FULL_OUTPUT = RUN_ROOT / "full"
subprocess.run([str(PYTHON), str(PROJECT / "scripts/run_baseline.py"),
                "--bundle", str(BUNDLE), "--output", str(FULL_OUTPUT), "--stage", "full",
                "--smoke-output", str(SMOKE_OUTPUT), "--batch-size", str(BATCH_SIZE)], check=True, cwd=PROJECT)

## 6. Read and preserve the baseline results

Report each benchmark separately. Accuracy includes all questions, including invalid and length-limited outputs. Wilson 95% intervals express uncertainty around baseline accuracy; the later baseline-versus-tuned comparison will use paired bootstrap intervals.

Timing is GPU-synchronized generation time divided across each batch. It is an amortized throughput measure, not individual request latency. Peak allocated GPU memory is recorded in the result manifest. A correct final answer does not prove every intermediate reasoning step is correct.

The report displays aggregate results. Avoid using test failures to repeatedly tune prompts or training; use validation for development decisions.

In [ ]:
SUMMARY = json.loads((FULL_OUTPUT / "summary.json").read_text())
if not SUMMARY["complete"] or SUMMARY["completed_predictions"] != 2 * TEST_QUESTIONS:
    raise RuntimeError("Evaluation is incomplete; resume it before treating these as final baseline scores.")
RESULT = json.loads((FULL_OUTPUT / "result_manifest.json").read_text())
for name, expected in RESULT["files_sha256"].items():
    actual = hashlib.sha256((FULL_OUTPUT / name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"Output checksum mismatch: {name}")
display(HTML((FULL_OUTPUT / "report.html").read_text()))
print("Predictions:", SUMMARY["completed_predictions"])
print("Peak allocated GPU memory (bytes):", RESULT["peak_cuda_allocated_bytes"])
ARCHIVE = Path(shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT))
print("Complete result bundle:", RUN_ROOT)
print("ZIP archive:", ARCHIVE)
print("Save Version with outputs. Keep this baseline bundle and the original data bundle for the fine-tuning comparison.")

## What happens next?

Save this Kaggle version with its outputs. `full/predictions.jsonl` contains raw completions and per-question scores; `metrics.csv` and `report.html` contain readable summaries. The `identity.json` and result manifest record the exact experiment.

Notebook 03 will train LoRA on the training split, select checkpoints on validation, and evaluate with the same protocol. This notebook has not trained the model.

To resume after a Kaggle session ends, retain/download the partial output directory, attach it to the next session along with Notebook 01's output, and run the same code revision on the same GPU type. The launcher restores a matching run directory; the evaluator refuses mismatched identities. Files under `/kaggle/working` are not guaranteed to survive an unsaved session.